In [1]:
# import libraries

#import libraries to conduct eda
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import os

# import for distance calculations
!pip install geopy
from geopy import distance
import math

# import scipy for statistical analysis in eda
#!pip install scipy
from scipy import stats

# import datetime to set timestamp on pdf report
from datetime import datetime, timedelta

print("All required libraries installed")



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
All required libraries installed


In [2]:
# data file names 

data_file_hurr = "../data/ibtracs.NA.list.v04r01.csv"

data_file_gas = "../data/aaa_fl_metros_wayback_2022_2025.csv"
#"../data/gas_coordinates.csv"

data_file_coast = "../data/Gulf_Coast_Coords.csv"

data_file_metro_loc = "../data/gas_coordinates.csv"

In [3]:
# import file loading functions
from file_load import *

In [4]:
pip install plotly


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px


# Importing libraries for KMeans and Heirarchical Clustering
from sklearn.cluster import KMeans
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

In [6]:
!pip install nbformat


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [7]:
# import data files

# import data files
hurr_file_chk = input('Is hurricane data file downloaded to data folder? (y/n) ')
if hurr_file_chk == 'y':
    hurr_data = df_import(data_file_hurr)
else:
    url = 'https://www.ncei.noaa.gov/data/international-best-track-archive-for-climate-stewardship-ibtracs/v04r01/access/csv/ibtracs.NA.list.v04r01.csv'
    hurr_data = pd.read_csv(url)
    
hurr_data.name = 'hurr_data'

gas_data = df_import(data_file_gas)
gas_data.name = 'gas_data'

coast_pt_data = df_import(data_file_coast)

metro_loc_data = coord_import(data_file_metro_loc)

/var/folders/ck/qyyf66gs1cn8jlq536xgp_500000gn/T/ipykernel_38544/1799337683.py:9: DtypeWarning: Columns (1,2,3,8,9,14,19,20,23,24,26,27,28,29,30,31,32,33,38,39,40,172,173) have mixed types. Specify dtype option on import or set low_memory=False.
  hurr_data = pd.read_csv(url)


Successfully loaded ../data/aaa_fl_metros_wayback_2022_2025.csv
Successfully loaded ../data/Gulf_Coast_Coords.csv
Successfully loaded ../data/gas_coordinates.csv


In [8]:
# import data cleaning functions
from clean import *

In [9]:
# Clean hurricane data and generate data frames for cleaned hurricane data, list of hurricanes, and summary list of hurricanes with maximum storm category observed
# Generates data frames with ALL data and a set of filtered data frames for "recent" (2022-2025)
hurr_data_redux, hurr_list, hurr_sum_list, hurr_data_redux_rec, hurr_list_rec, hurr_sum_list_rec, hurr_data_redux_rec_hur = hurr_clean(hurr_data)

# Name output data frames
hurr_data_redux.name = 'hurr_data_redux'
hurr_list.name = 'hurr_list'
hurr_sum_list.name = 'hurr_sum_list'
hurr_data_redux_rec.name = 'hurr_data_redux_rec'
hurr_list_rec.name = 'hurr_list_rec'
hurr_sum_list_rec.name = 'hurr_sum_list_rec'
hurr_data_redux_rec_hur.name = 'hurr_data_redux_rec_hur'

Cleaning hurr_data
          Unit
SID           
SEASON    Year
NUMBER        
BASIN         
SUBBASIN      
(174, 1)
Data table columns reduced to 47
Hurricane data cleaning complete


In [10]:
# List metro areas of interest
# all FL cities selected to represent the perimeter

metro_all = ['Pensacola', 'Tallahassee', 'Tampa-St. Petersburg-Clearwater', 'Fort Myers-Cape Coral', 'Miami','West Palm Beach-Boca Raton','Melbourne-Titusville','Daytona Beach','Jacksonville']

# East coast of FL
metro_east = ['Miami','West Palm Beach-Boca Raton','Melbourne-Titusville','Daytona Beach','Jacksonville']

# West coast of FL (exl panhandle)
metro_west = ['Tampa-St. Petersburg-Clearwater', 'Fort Myers-Cape Coral']

# Panhandle of FL
metro_ph = ['Pensacola', 'Tallahassee']

# Gulf Coast of FL (West + Panhandle)
metro_gulf = ['Pensacola', 'Tallahassee', 'Tampa-St. Petersburg-Clearwater', 'Fort Myers-Cape Coral']


In [11]:
# Clean gas station data and generate data frame for cleaned gas station data
gas_data_redux = gas_clean(gas_data)
gas_data_redux.name = 'gas_data_redux'

Cleaning gas_data
Gas data cleaning complete


In [12]:
# List of hurricanes from 2022 on
hurr_can_list = hurr_sum_list_rec[hurr_sum_list_rec["MAX CAT"] > 0]
hurr_can_list

,SID,SEASON,NAME,MAX CAT,START DATE,END DATE,STORM DURATION
2224,2022179N08310,2022,BONNIE,3,2022-06-27 18:00:00,2022-07-11 00:00:00,13
2227,2022244N38313,2022,DANIELLE,1,2022-08-31 12:00:00,2022-09-15 18:00:00,15
2228,2022246N18301,2022,EARL,2,2022-09-02 18:00:00,2022-09-15 12:00:00,12
2229,2022257N16312,2022,FIONA,4,2022-09-14 06:00:00,2022-09-27 18:00:00,13
2231,2022266N12294,2022,IAN,5,2022-09-22 18:00:00,2022-10-01 06:00:00,8
2235,2022280N11294,2022,JULIA,1,2022-10-06 12:00:00,2022-10-10 12:00:00,4
2237,2022304N16287,2022,LISA,1,2022-10-30 18:00:00,2022-11-05 06:00:00,5
2238,2022304N34296,2022,MARTIN,1,2022-10-30 18:00:00,2022-11-04 18:00:00,5
2239,2022311N21293,2022,NICOLE,1,2022-11-06 12:00:00,2022-11-11 18:00:00,5
2244,2023193N37305,2023,DON,1,2023-07-11 12:00:00,2023-07-25 18:00:00,14


In [13]:
# Additional cleaning of hurricane data set - removing additional unnecessary columns 
col_to_remove1 = ['SEASON_y','NUMBER','BASIN','SUBBASIN','NAME_y','LAT','LON','TRACK_TYPE','IFLAG','USA_AGENCY','USA_ATCF_ID']
col_to_remove2 = ['USA_R34_NE','USA_R34_SE','USA_R34_SW','USA_R34_NW','USA_R50_NE','USA_R50_SE','USA_R50_SW','USA_R50_NW','USA_R64_NE','USA_R64_SE','USA_R64_SW','USA_R64_NW','USA_SEARAD_NE','USA_SEARAD_SE','USA_SEARAD_SW','USA_SEARAD_NW']
hurr_mod_data = hurr_data_redux_rec_hur.drop(col_to_remove1, axis = 1)
hurr_mod_data.drop(col_to_remove2, axis = 1, inplace = True)
# fix/rename columns
hurr_mod_data.rename(columns={'SEASON_x': 'SEASON', 'NAME_x': 'NAME'}, inplace=True)
hurr_mod_data.columns

Index(['SID', 'SEASON', 'NAME', 'MAX CAT', 'START DATE', 'END DATE',
       'STORM DURATION', 'ISO_TIME', 'NATURE', 'DIST2LAND', 'LANDFALL',
       'USA_LAT', 'USA_LON', 'USA_RECORD', 'USA_STATUS', 'USA_WIND',
       'USA_PRES', 'USA_SSHS', 'USA_POCI', 'USA_ROCI', 'USA_RMW', 'USA_EYE',
       'USA_GUST', 'USA_SEAHGT', 'STORM_SPEED', 'STORM_DIR', 'DATE'],
      dtype='object')

In [14]:
hurr_mod_data.head()

,SID,SEASON,NAME,MAX CAT,START DATE,END DATE,STORM DURATION,ISO_TIME,NATURE,DIST2LAND,...,USA_SSHS,USA_POCI,USA_ROCI,USA_RMW,USA_EYE,USA_GUST,USA_SEAHGT,STORM_SPEED,STORM_DIR,DATE
0,2022179N08310,2022,BONNIE,3,2022-06-27 18:00:00,2022-07-11,13,2022-06-27 18:00:00,DS,417,...,-3,1012.0,150.0,120.0,NaN,40,NaN,17.0,285.0,2022-06-27
1,2022179N08310,2022,BONNIE,3,2022-06-27 18:00:00,2022-07-11,13,2022-06-27 21:00:00,DS,394,...,-3,1012.0,150.0,120.0,NaN,,NaN,17.0,285.0,2022-06-27
2,2022179N08310,2022,BONNIE,3,2022-06-27 18:00:00,2022-07-11,13,2022-06-28 00:00:00,DS,379,...,-3,1012.0,150.0,120.0,NaN,45,12.0,19.0,280.0,2022-06-28
3,2022179N08310,2022,BONNIE,3,2022-06-27 18:00:00,2022-07-11,13,2022-06-28 03:00:00,DS,353,...,-3,1012.0,150.0,120.0,NaN,,12.0,20.0,280.0,2022-06-28
4,2022179N08310,2022,BONNIE,3,2022-06-27 18:00:00,2022-07-11,13,2022-06-28 06:00:00,DS,358,...,-3,1012.0,150.0,120.0,NaN,45,12.0,22.0,280.0,2022-06-28


In [15]:
gas_data_redux=gas_data_redux[gas_data_redux['metro']=='Miami']
gas_data_redux.head()

,metro,date,regular,mid,premium,diesel,year,month,date_key,date_time_key
14400,Miami,2022-01-21,3.2310,3.598000,3.853000,3.545,2022,1,2022-01-21,2022-01-21 12:00:00
14401,Miami,2022-01-22,3.2435,3.605333,3.861833,3.562,2022,1,2022-01-22,2022-01-22 12:00:00
14402,Miami,2022-01-23,3.2560,3.612667,3.870667,3.579,2022,1,2022-01-23,2022-01-23 12:00:00
14403,Miami,2022-01-24,3.2685,3.620000,3.879500,3.596,2022,1,2022-01-24,2022-01-24 12:00:00
14404,Miami,2022-01-25,3.2810,3.627333,3.888333,3.613,2022,1,2022-01-25,2022-01-25 12:00:00


In [16]:
hurr_mod_data['DATE']=pd.to_datetime(hurr_mod_data['DATE'])
gas_data_redux['date']=pd.to_datetime(gas_data_redux['date'])


In [17]:
merged = pd.merge(hurr_mod_data, gas_data_redux, left_on='DATE',right_on = 'date', how='inner')

In [18]:
merged.head()

,SID,SEASON,NAME,MAX CAT,START DATE,END DATE,STORM DURATION,ISO_TIME,NATURE,DIST2LAND,...,metro,date,regular,mid,premium,diesel,year,month,date_key,date_time_key
0,2022179N08310,2022,BONNIE,3,2022-06-27 18:00:00,2022-07-11,13,2022-06-27 18:00:00,DS,417,...,Miami,2022-06-27,4.757,5.167,5.453,5.692,2022,6,2022-06-27,2022-06-27 12:00:00
1,2022179N08310,2022,BONNIE,3,2022-06-27 18:00:00,2022-07-11,13,2022-06-27 21:00:00,DS,394,...,Miami,2022-06-27,4.757,5.167,5.453,5.692,2022,6,2022-06-27,2022-06-27 12:00:00
2,2022179N08310,2022,BONNIE,3,2022-06-27 18:00:00,2022-07-11,13,2022-06-28 00:00:00,DS,379,...,Miami,2022-06-28,4.728,5.149,5.424,5.672,2022,6,2022-06-28,2022-06-28 12:00:00
3,2022179N08310,2022,BONNIE,3,2022-06-27 18:00:00,2022-07-11,13,2022-06-28 03:00:00,DS,353,...,Miami,2022-06-28,4.728,5.149,5.424,5.672,2022,6,2022-06-28,2022-06-28 12:00:00
4,2022179N08310,2022,BONNIE,3,2022-06-27 18:00:00,2022-07-11,13,2022-06-28 06:00:00,DS,358,...,Miami,2022-06-28,4.728,5.149,5.424,5.672,2022,6,2022-06-28,2022-06-28 12:00:00


In [19]:
#merged = merged.drop(columns = ['USA_SEAHGT','date_time_key'])


In [20]:
start_prices = merged.sort_values('date').groupby('NAME').first().reset_index()
start_prices = start_prices[['NAME','regular']].rename(columns = {'regular':'start_price'})

end_prices = merged.sort_values('date').groupby('NAME').last().reset_index()
end_prices = end_prices[['NAME','regular']].rename(columns = {'regular':'end_price'})

In [21]:
storm_summary = merged.groupby('NAME').agg({
    'START DATE':'first',
    'END DATE':'first',
    'MAX CAT':'max',
    'STORM DURATION':'max'
}).reset_index()

In [74]:
storm_summary = storm_summary.merge(start_prices, on='NAME')
storm_summary = storm_summary.merge(end_prices, on='NAME')
storm_summary

,NAME,START DATE,END DATE,MAX CAT,STORM DURATION,start_price_x,end_price_x,gas_pct_change,start_month,end_month,start_price_y,end_price_y
0,BERYL,2024-06-28 12:00:00,2024-07-11 12:00:00,5,13,3.396000,3.489000,2.738516,6,7,3.396000,3.489000
1,BONNIE,2022-06-27 18:00:00,2022-07-11 00:00:00,3,13,4.757000,4.500933,-5.382944,6,7,4.757000,4.500933
2,DANIELLE,2022-08-31 12:00:00,2022-09-15 18:00:00,1,15,3.679667,3.505000,-4.746807,8,9,3.679667,3.505000
3,DEBBY,2024-08-02 12:00:00,2024-08-10 18:00:00,1,8,3.417667,3.369000,-1.423973,8,8,3.417667,3.369000
4,DON,2023-07-11 12:00:00,2023-07-25 18:00:00,1,14,3.320610,3.315829,-0.143964,7,7,3.320610,3.315829
5,EARL,2022-09-02 18:00:00,2022-09-15 12:00:00,2,12,3.656378,3.505000,-4.140102,9,9,3.656378,3.505000
6,ERIN,2025-08-11 00:00:00,2025-08-27 18:00:00,5,16,2.986000,3.080000,3.148024,8,8,2.986000,3.080000
7,ERNESTO,2024-08-11 18:00:00,2024-08-20 18:00:00,2,9,3.344000,3.328000,-0.478469,8,8,3.344000,3.328000
8,FIONA,2022-09-14 06:00:00,2022-09-27 18:00:00,4,13,3.516644,3.453000,-1.809806,9,9,3.516644,3.453000
9,FRANCINE,2024-09-08 18:00:00,2024-09-14 00:00:00,2,5,3.166000,3.134000,-1.010739,9,9,3.166000,3.134000


In [23]:
storm_summary['gas_pct_change']=((storm_summary['end_price']-storm_summary['start_price'])/storm_summary['start_price'])*100
print(storm_summary.columns)

Index(['NAME', 'START DATE', 'END DATE', 'MAX CAT', 'STORM DURATION',
       'start_price', 'end_price', 'gas_pct_change'],
      dtype='object')


In [24]:
storm_summary['start_month']=pd.to_datetime(storm_summary['START DATE']).dt.month
storm_summary['end_month']=pd.to_datetime(storm_summary['END DATE']).dt.month

In [25]:
storm_df = storm_summary[[
    'NAME',
    'start_month',
    'end_month',
    'MAX CAT',
    'STORM DURATION',
    'gas_pct_change',
]]

In [90]:
storm_df

storm_df.to_csv('storm_df.csv')


In [27]:
print(storm_df['gas_pct_change'].max())
print(storm_df['gas_pct_change'].min())


3.4277719482336435
-5.872965209064813


In [28]:
#Create bins for storm data
df = storm_df.copy()

df['category']='cat_'+df['MAX CAT'].astype(str)


#duration min = 3, max = 21, categorize into short (0,7), medium (7,14), long storm (14,21)
df['duration_bin']=pd.cut(
    df['STORM DURATION'],
    bins=[0,7,14,22],
    labels = ['short storm length','medium storm length','long storm length']
)

df['gas_pct_change_bin']=pd.cut(
    df['gas_pct_change'],
    bins=[-10,-0.02,0.02,10],
    labels=['decrease','no change','increase'],
)

df['start_month']='start month' + df['start_month'].astype(str)
df['end_month']='end month' + df['end_month'].astype(str)

df = df.dropna(subset=['start_month',
    'end_month',
    'category',
    'duration_bin',
    'gas_pct_change_bin'])

In [88]:
#Investigate binning for storm duration: want more storms in short and medium duration category
count = (df['STORM DURATION']<7).sum()
print(count)
count2 = ((df['STORM DURATION']>=7) & (df['STORM DURATION']<14)).sum()
print(count2)
count3 = ((df['STORM DURATION']>=14) & (df['STORM DURATION']<22)).sum()
print(count3)

8
20
4


In [30]:
transactions = df[[
    'start_month',
    'category',
    'duration_bin',
    'gas_pct_change_bin'
]].values.tolist()

In [31]:
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth
from mlxtend.frequent_patterns import association_rules



In [32]:
te = TransactionEncoder()
te_array = te.fit(transactions).transform(transactions)

df_encoded = pd.DataFrame(te_array, columns=te.columns_)


In [33]:
frequent_itemsets = fpgrowth(df_encoded,min_support = 0.15, use_colnames = True)


In [34]:
rules = association_rules(frequent_itemsets,metric = 'confidence', min_threshold = 0.6)

In [35]:
rules

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,(medium storm length),(decrease),0.59375,0.59375,0.46875,0.789474,1.329640,1.0,0.116211,1.929688,0.610256,0.652174,0.481781,0.789474
1,(decrease),(medium storm length),0.59375,0.59375,0.46875,0.789474,1.329640,1.0,0.116211,1.929688,0.610256,0.652174,0.481781,0.789474
2,(cat_1),(increase),0.31250,0.40625,0.18750,0.600000,1.476923,1.0,0.060547,1.484375,0.469697,0.352941,0.326316,0.530769
3,"(increase, cat_1)",(short storm length),0.18750,0.31250,0.15625,0.833333,2.666667,1.0,0.097656,4.125000,0.769231,0.454545,0.757576,0.666667
4,"(increase, short storm length)",(cat_1),0.25000,0.31250,0.15625,0.625000,2.000000,1.0,0.078125,1.833333,0.666667,0.384615,0.454545,0.562500
5,"(cat_1, short storm length)",(increase),0.15625,0.40625,0.15625,1.000000,2.461538,1.0,0.092773,inf,0.703704,0.384615,1.000000,0.692308
6,(start month8),(decrease),0.18750,0.59375,0.15625,0.833333,1.403509,1.0,0.044922,2.437500,0.353846,0.250000,0.589744,0.548246
7,(start month9),(medium storm length),0.40625,0.59375,0.31250,0.769231,1.295547,1.0,0.071289,1.760417,0.384211,0.454545,0.431953,0.647773
8,(start month9),(decrease),0.40625,0.59375,0.28125,0.692308,1.165992,1.0,0.040039,1.320312,0.239766,0.391304,0.242604,0.582996
9,"(start month9, decrease)",(medium storm length),0.28125,0.59375,0.25000,0.888889,1.497076,1.0,0.083008,3.656250,0.461957,0.400000,0.726496,0.654971


In [36]:
rules_gas_increase = rules[rules['consequents'].apply(
    lambda x: 'increase' in str(x)
)]
rules_gas_increase

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
2,(cat_1),(increase),0.31250,0.40625,0.18750,0.600,1.476923,1.0,0.060547,1.484375,0.469697,0.352941,0.326316,0.530769
5,"(cat_1, short storm length)",(increase),0.15625,0.40625,0.15625,1.000,2.461538,1.0,0.092773,inf,0.703704,0.384615,1.000000,0.692308
19,(short storm length),(increase),0.31250,0.40625,0.25000,0.800,1.969231,1.0,0.123047,2.968750,0.715909,0.533333,0.663158,0.707692
21,(start month10),(increase),0.25000,0.40625,0.15625,0.625,1.538462,1.0,0.054688,1.583333,0.466667,0.312500,0.368421,0.504808


In [37]:
rule = rules.iloc[19]

antecedents = list(rule['antecedents'])
consequents = list(rule['consequents'])
items = antecedents + consequents

matching_rows = df_encoded[list(items)].all(axis=1)
storms = df.loc[matching_rows.values]

storms

,NAME,start_month,end_month,MAX CAT,STORM DURATION,gas_pct_change,category,duration_bin,gas_pct_change_bin
12,HELENE,start month9,end month9,4,5,2.113585,cat_4,short storm length,increase
13,HUMBERTO,start month9,end month10,5,7,1.057851,cat_5,short storm length,increase
18,JULIA,start month10,end month10,1,4,0.501975,cat_1,short storm length,increase
22,LISA,start month10,end month11,1,5,3.331350,cat_1,short storm length,increase
24,MARTIN,start month10,end month11,1,5,2.974420,cat_1,short storm length,increase
27,NICOLE,start month11,end month11,1,5,2.236513,cat_1,short storm length,increase
29,OSCAR,start month10,end month10,1,3,0.372590,cat_1,short storm length,increase
30,RAFAEL,start month11,end month11,3,6,1.347798,cat_3,short storm length,increase


In [89]:
rules_gas_decrease = rules[rules['consequents'].apply(
    lambda x: 'decrease' in str(x)
)]
rules_gas_decrease

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,(medium storm length),(decrease),0.59375,0.59375,0.46875,0.789474,1.329640,1.0,0.116211,1.929688,0.610256,0.652174,0.481781,0.789474
6,(start month8),(decrease),0.18750,0.59375,0.15625,0.833333,1.403509,1.0,0.044922,2.437500,0.353846,0.250000,0.589744,0.548246
8,(start month9),(decrease),0.40625,0.59375,0.28125,0.692308,1.165992,1.0,0.040039,1.320312,0.239766,0.391304,0.242604,0.582996
10,"(medium storm length, start month9)",(decrease),0.31250,0.59375,0.25000,0.800000,1.347368,1.0,0.064453,2.031250,0.375000,0.380952,0.507692,0.610526
11,(start month9),"(medium storm length, decrease)",0.40625,0.46875,0.25000,0.615385,1.312821,1.0,0.059570,1.381250,0.401316,0.400000,0.276018,0.574359
12,(cat_2),(decrease),0.21875,0.59375,0.18750,0.857143,1.443609,1.0,0.057617,2.843750,0.393333,0.300000,0.648352,0.586466
14,"(medium storm length, cat_2)",(decrease),0.18750,0.59375,0.15625,0.833333,1.403509,1.0,0.044922,2.437500,0.353846,0.250000,0.589744,0.548246
16,(cat_2),"(medium storm length, decrease)",0.21875,0.46875,0.15625,0.714286,1.523810,1.0,0.053711,1.859375,0.440000,0.294118,0.462185,0.523810
17,(cat_4),(decrease),0.18750,0.59375,0.15625,0.833333,1.403509,1.0,0.044922,2.437500,0.353846,0.250000,0.589744,0.548246
